### Installation & Environment Setup

Run the following command to install required dependencies:

`ash
pip install matplotlib numpy
`

# 📊 Veritas AI — Frontier Detection Analysis & Figures

> [!WARNING]
> **UNRUN Template**: This notebook has not been executed and contains no cached outputs or hard-coded metrics.
> All numbers visualized by this notebook must originate from empirical JSON results generated by `python scripts/eval_frontier.py` and stored in `data/eval/results/*.json`.
> If result files or specific keys have not yet been produced by the evaluation pipeline, cells gracefully display **"TBD (from data/reports/FRONTIER_DETECTION_REPORT.md)"**.

In [ ]:
# Cell 1: Setup, dependencies, and path configuration
import os
import glob
import json
import numpy as np
import matplotlib.pyplot as plt

# Locate results directory (handles execution from repo root or notebooks/)
CWD = os.path.abspath(os.getcwd())
REPO_ROOT = CWD if os.path.exists(os.path.join(CWD, 'scripts')) else os.path.dirname(CWD)
RESULTS_DIR = os.path.join(REPO_ROOT, 'data', 'eval', 'results')

print(f'Repository root: {REPO_ROOT}')
print(f'Results directory: {RESULTS_DIR}')

# Matplotlib formatting for academic publication quality
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 10,
    'axes.labelsize': 11,
    'axes.titlesize': 12,
    'figure.titlesize': 13,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'grid.linestyle': '--',
})

## 1. Load Evaluation Results from `data/eval/results/*.json`

The schema produced by `scripts/eval_frontier.py` contains top-level keys:
- `detector` (`str`): Identifier of the evaluated detector (`shipped`, `hc3_roberta`, `binoculars`, etc.)
- `split` (`str`): Dataset split (`dev` or `locked`)
- `threshold_1pct_fpr_dev` (`float`): Threshold calibrated at 1% FPR on dev clean-human texts
- `auroc_pooled` (`float`): Area under ROC curve (AI vs. clean human)
- `tpr_pooled`, `tpr_raw`, `tpr_attacked` (`dict`): Rate and Wilson 95% CI `{"rate", "lo", "hi", "k", "n"}`
- `tpr_by_generator_family` (`dict[str, dict]`): TPR breakdown by generator and attack family
- `fpr_clean_human`, `fpr_esl`, `fpr_native`, `fpr_by_genre` (`dict`): Realized FPR breakdowns

In [ ]:
# Cell 2: Ingest results from data/eval/results/*.json
def load_all_evaluation_results(results_dir):
    pattern = os.path.join(results_dir, '*.json')
    files = glob.glob(pattern)
    eval_records = []
    if not files:
        print(f'TBD (from data/reports/FRONTIER_DETECTION_REPORT.md): No JSON files found in {results_dir}.')
        print('To generate results, run:')
        print('  python scripts/eval_frontier.py --split dev --detectors shipped hc3_roberta binoculars --out data/eval/results/dev_baselines.json')
        return eval_records
    for fp in sorted(files):
        try:
            with open(fp, 'r', encoding='utf-8') as f:
                doc = json.load(f)
            # Doc can be a single result dict or {'results': [...], 'args': ...}
            if isinstance(doc, dict) and 'results' in doc:
                for r in doc['results']:
                    r['_source_file'] = os.path.basename(fp)
                    eval_records.append(r)
            elif isinstance(doc, dict):
                doc['_source_file'] = os.path.basename(fp)
                eval_records.append(doc)
            print(f'Loaded {len(eval_records)} record(s) from {os.path.basename(fp)}')
        except Exception as e:
            print(f'Error reading {fp}: {e}')
    return eval_records

records = load_all_evaluation_results(RESULTS_DIR)
print(f'Total evaluation records available: {len(records)}')

## 2. Figure 1: TPR @ 1% FPR Across Generators and Attack Families

Visualizes True Positive Rate with Wilson 95% Confidence Interval error bars for each frontier generator (GPT-4o, Claude 3.5, Gemini 1.5, DeepSeek-V3, etc.) across attack families:
- **Raw (None)**: Unmodified direct model output
- **A1**: Single-pass LLM paraphrase
- **A2**: Iterative multi-pass LLM rewrite
- **A3**: Humanizer prompt evasion
- **A4-A7**: Algorithmic attacks (token insertions, typo injectors, zero-width spaces)

In [ ]:
# Cell 3: Plot TPR with Wilson 95% CIs per generator x attack family
def plot_tpr_by_generator_family(records, target_split='dev'):
    split_records = [r for r in records if r.get('split') == target_split]
    if not split_records:
        print(f'TBD (from data/reports/FRONTIER_DETECTION_REPORT.md): No {target_split} split records found.')
        return

    for rec in split_records:
        det_name = rec.get('detector', 'Unknown Detector')
        gen_family = rec.get('tpr_by_generator_family', {})
        if not gen_family:
            print(f'TBD (from data/reports/FRONTIER_DETECTION_REPORT.md): tpr_by_generator_family missing for {det_name}.')
            continue

        labels = list(gen_family.keys())
        rates = [gen_family[k]['rate'] * 100 for k in labels]
        err_lo = [(gen_family[k]['rate'] - gen_family[k]['lo']) * 100 for k in labels]
        err_hi = [(gen_family[k]['hi'] - gen_family[k]['rate']) * 100 for k in labels]

        fig, ax = plt.subplots(figsize=(10, max(4, len(labels) * 0.35)))
        y_pos = np.arange(len(labels))
        ax.errorbar(rates, y_pos, xerr=[err_lo, err_hi], fmt='o', color='#2563eb',
                    ecolor='#93c5fd', elinewidth=2, capsize=4, markersize=6)
        ax.set_yticks(y_pos)
        ax.set_yticklabels(labels)
        ax.set_xlabel('TPR @ 1% FPR (%) [Wilson 95% CI]')
        ax.set_title(f'Detection Rate by Generator × Attack Family — {det_name} ({target_split})')
        ax.set_xlim(0, 105)
        ax.axvline(100, color='gray', linestyle=':', alpha=0.6)
        plt.tight_layout()
        plt.show()

plot_tpr_by_generator_family(records, 'dev')

## 3. Figure 2: Realized False Positive Rates (FPR) by Genre and ESL vs. Native

Audit of false positives on human-authored controls. The decision threshold is calibrated to yield $\le 1.0\%$ FPR on dev clean-human texts.
This figure audits fairness across ESL students vs. native English writers, as well as cross-genre variance (academic essays, news, creative prose, technical discourse).

In [ ]:
# Cell 4: Plot realized FPR by genre and ESL vs native writers
def plot_fpr_audit(records, target_split='dev'):
    split_records = [r for r in records if r.get('split') == target_split]
    if not split_records:
        print(f'TBD (from data/reports/FRONTIER_DETECTION_REPORT.md): No {target_split} records available for FPR audit.')
        return

    fig, (ax_demo, ax_genre) = plt.subplots(1, 2, figsize=(12, 5))
    colors = ['#10b981', '#f59e0b', '#6366f1', '#ec4899']

    # Panel A: Demographic audit (Clean Human, ESL, Native)
    demo_labels = ['Clean Human', 'ESL (Learner)', 'Native']
    x = np.arange(len(demo_labels))
    width = 0.8 / max(1, len(split_records))

    for i, rec in enumerate(split_records):
        det_name = rec.get('detector', f'Det {i+1}')
        rates = [
            rec.get('fpr_clean_human', {}).get('rate', 0.0) * 100,
            rec.get('fpr_esl', {}).get('rate', 0.0) * 100,
            rec.get('fpr_native', {}).get('rate', 0.0) * 100,
        ]
        offset = (i - len(split_records) / 2 + 0.5) * width
        ax_demo.bar(x + offset, rates, width, label=det_name, color=colors[i % len(colors)], alpha=0.85)

    ax_demo.axhline(1.0, color='#ef4444', linestyle='--', linewidth=1.5, label='1.0% Target')
    ax_demo.set_xticks(x)
    ax_demo.set_xticklabels(demo_labels)
    ax_demo.set_ylabel('Realized FPR (%)')
    ax_demo.set_title(f'ESL vs. Native Human FPR ({target_split})')
    ax_demo.legend(loc='upper right')

    # Panel B: Genre audit
    first_rec = split_records[0]
    by_genre = first_rec.get('fpr_by_genre', {})
    if by_genre:
        genres = list(by_genre.keys())
        g_rates = [by_genre[g]['rate'] * 100 for g in genres]
        ax_genre.barh(genres, g_rates, color='#0284c7', alpha=0.85)
        ax_genre.axvline(1.0, color='#ef4444', linestyle='--', linewidth=1.5)
        ax_genre.set_xlabel('Realized FPR (%)')
        ax_genre.set_title(f'FPR by Genre — {first_rec.get("detector", "")} ({target_split})')
    else:
        ax_genre.text(0.5, 0.5, 'TBD: fpr_by_genre pending evaluation', ha='center', va='center')

    plt.tight_layout()
    plt.show()

plot_fpr_audit(records, 'dev')

## 4. Figure 3: Dev Split vs. Locked Test Split Generalization

Comparison of detection metrics between the development split (`dev`) and the unseen held-out test split (`locked`).
- Validates that detectors do not overfit to development prompts or known generator distributions.
- Compares AUROC, Pooled TPR @ 1% FPR, and ESL FPR.

In [ ]:
# Cell 5: Dev vs. Locked Split comparison
def plot_dev_vs_locked(records):
    dev_recs = {r['detector']: r for r in records if r.get('split') == 'dev' and 'detector' in r}
    locked_recs = {r['detector']: r for r in records if r.get('split') == 'locked' and 'detector' in r}

    common_detectors = [d for d in dev_recs if d in locked_recs]
    if not common_detectors:
        print('TBD (from data/reports/FRONTIER_DETECTION_REPORT.md): Locked split baseline evaluation not yet executed.')
        print('The locked split is evaluated via:')
        print('  python scripts/eval_frontier.py --split locked --detectors shipped hc3_roberta binoculars --out data/eval/results/locked_baselines.json')
        return

    fig, ax = plt.subplots(figsize=(8, 4.5))
    x = np.arange(len(common_detectors))
    width = 0.35

    dev_tpr = [dev_recs[d].get('tpr_pooled', {}).get('rate', 0.0) * 100 for d in common_detectors]
    locked_tpr = [locked_recs[d].get('tpr_pooled', {}).get('rate', 0.0) * 100 for d in common_detectors]

    rects1 = ax.bar(x - width/2, dev_tpr, width, label='Dev Split', color='#3b82f6')
    rects2 = ax.bar(x + width/2, locked_tpr, width, label='Locked Split', color='#10b981')

    ax.set_ylabel('Pooled TPR @ 1% FPR (%)')
    ax.set_title('Generalization: Dev vs. Locked Test Split Performance')
    ax.set_xticks(x)
    ax.set_xticklabels(common_detectors)
    ax.set_ylim(0, 105)
    ax.legend()

    plt.tight_layout()
    plt.show()

plot_dev_vs_locked(records)

## 5. Structured Metric Summary (Report-Ready Markdown)

Formats loaded records into the exact table layout specified in `EVAL_REPORT.md`.
All missing entries are marked `TBD`.

In [ ]:
# Cell 6: Generate Markdown summary table
def print_markdown_summary(records):
    if not records:
        print('TBD (from data/reports/FRONTIER_DETECTION_REPORT.md): No evaluation records found.')
        return

    headers = ['Detector', 'Split', 'AUROC', 'TPR @ 1% FPR [95% CI]', 'Realized FPR', 'ESL FPR']
    rows = []
    for r in records:
        det = r.get('detector', 'TBD')
        split = r.get('split', 'TBD')
        auroc = f"{r.get('auroc_pooled', 0.0):.4f}" if 'auroc_pooled' in r else 'TBD'
        tpr = r.get('tpr_pooled', {})
        tpr_str = f"{tpr.get('rate', 0.0)*100:.1f}% [{tpr.get('lo', 0.0)*100:.1f}-{tpr.get('hi', 0.0)*100:.1f}]" if tpr else 'TBD'
        fpr = r.get('fpr_clean_human', {})
        fpr_str = f"{fpr.get('rate', 0.0)*100:.2f}%" if fpr else 'TBD'
        esl = r.get('fpr_esl', {})
        esl_str = f"{esl.get('rate', 0.0)*100:.2f}%" if esl else 'TBD'
        rows.append([det, split, auroc, tpr_str, fpr_str, esl_str])

    print('| ' + ' | '.join(headers) + ' |')
    print('| ' + ' | '.join(['---'] * len(headers)) + ' |')
    for row in rows:
        print('| ' + ' | '.join(row) + ' |')

print_markdown_summary(records)